In [ ]:
import sys

sys.path.append("..")

import matplotlib.pyplot as plt
import numpy as np
import random
from pulp import *

In [ ]:
from candidate.candidate import Candidate
from voter.voter import Voter
from election.election import Election
from election.result import ElectionResult
from analysis.analyzer import ResultsAnalyzer

from strategy.strategy import VotingStrategy
from strategy.plurality import PluralityStrategy
from strategy.borda import BordaCountStrategy
from strategy.veto import VetoStrategy

from visualization.util import (
    plot,
    plot_results,
    random_2d_points,
    random_2d_point,
    plot_winner_distance_histogram,
    plot_lp_result,
)

from calculation.calculation import find_farthest_pair, find_farthest_triple

from lp.lp import MarginalLpModel, PermutationLpModel

In [ ]:
def verify_lp_winners(voter_positions, candidates, strategies, winners):
    """Check that voters generated by the LP actually elect the target winners."""
    lp_positions = voter_positions[
        ~np.isnan(voter_positions[:, 0])
    ]  # drop NaN rows (rankings not realizable in 2D)
    lp_voters = [Voter(position=p) for p in lp_positions]
    lp_result = Election(candidates=candidates, voters=lp_voters).compare_strategies(
        strategies=strategies
    )

    print(f"voters: {len(lp_voters)}\n")
    print(f"{'rule':<12} {'target':>7} {'actual':>7}  ok")
    all_ok = True
    for s in strategies:
        target_idx = winners[s.key]
        actual_idx = candidates.index(lp_result.winner(s))
        ok = target_idx == actual_idx
        all_ok &= ok
        print(f"{s.key:<12} {target_idx:>7} {actual_idx:>7}  {'OK' if ok else 'X'}")

    assert all_ok, "LP model does NOT reproduce the target winners!"
    print("\nAll winners match — LP model is correct.")

In [ ]:
N_VOTERS = 100
N_CANDIDATES = 3